# Preprocesamiento

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Datos

Separamos el archivo de entrenamiento en entrenamiento y validación con `split_train_validation`, y cargamos la prueba con `load_test`.

Todo lo que el preprocesamiento aprende de los datos (categorías conocidas, medias y desviaciones) se ajusta **solo con la parte de entrenamiento**. La validación y la prueba solo se transforman, para comprobar que el preprocesamiento funciona con flujos que no vio.

In [ ]:
from threat_detector.dataset import TARGET, load_test, load_train
from threat_detector.ml.preprocessing import split_train_validation

train_flows = load_train()
train_part, validation_part = split_train_validation(train_flows)
test_flows = load_test()

for name, flows in (("Entrenamiento", train_part), ("Validación", validation_part), ("Prueba", test_flows)):
    print(f"{name}: {len(flows):,} flujos x {flows.shape[1]} columnas; ataques = {flows[TARGET].mean():.1%}")

## 2. Fracción mínima de categorías

El análisis exploratorio conservó las categorías con al menos `MIN_CATEGORY_COUNT` (1.000) flujos en el archivo de entrenamiento completo. Como aquí se ajusta solo con la parte de entrenamiento (el 80 %), el mínimo se expresa como proporción: `MIN_CATEGORY_FRACTION`. Con una cuenta fija de 1.000, `sctp` y `pop3` quedarían por debajo y cambiarían las categorías decididas.

Mostramos la fracción, el corte que da en la parte de entrenamiento y cómo quedan frente a él las tres categorías más cercanas al límite.

In [ ]:
import pandas as pd

from threat_detector.config import MIN_CATEGORY_COUNT, MIN_CATEGORY_FRACTION

cutoff = MIN_CATEGORY_FRACTION * len(train_part)
print(f"MIN_CATEGORY_FRACTION = {MIN_CATEGORY_FRACTION:.5f} ({MIN_CATEGORY_FRACTION:.2%})")
print(f"Corte en la parte de entrenamiento: {cutoff:.0f} flujos (antes: {MIN_CATEGORY_COUNT})")

near_limit = [("proto", "sctp"), ("service", "pop3"), ("service", "ssh")]
pd.DataFrame(
    [
        {
            "variable": col,
            "categoría": category,
            "archivo completo": (train_flows[col] == category).sum(),
            "parte de entrenamiento": (train_part[col] == category).sum(),
            "¿pasa el corte?": (train_part[col] == category).sum() >= cutoff,
        }
        for col, category in near_limit
    ]
)

## 3. Columnas eliminadas, TTL y `log1p`

Tres listas de `preprocessing.py` guardan decisiones del análisis exploratorio:
- `DROPPED_COLS`: las 7 columnas que no se usan como variables de entrada (decisiones 4, 10, 11, 12 y 15).
- `TTL_COLS`: las 3 variables de TTL, que el método M3 deja fuera (decisión 14).
- `LOG_COLS`: las 21 variables a las que se aplica `log1p` (decisiones 7 y 10).

Comprobamos sus tamaños, que todas son variables de entrada y que no se cruzan donde no deben. Además, recalculamos la regla de `log1p` con el archivo de entrenamiento completo, igual que el análisis exploratorio (más de 20 valores distintos, asimetría mayor que 1 y `log1p` la reduce a menos de la mitad): debe dar `LOG_COLS` más `stcpb` y `dtcpb`, que se eliminaron después por la decisión 10.

In [ ]:
import numpy as np

from threat_detector.dataset import CATEGORICAL_COLS, INPUT_COLS
from threat_detector.ml.preprocessing import DROPPED_COLS, LOG_COLS, TTL_COLS

print(f"DROPPED_COLS: {len(DROPPED_COLS)}; TTL_COLS: {len(TTL_COLS)}; LOG_COLS: {len(LOG_COLS)}")
print(f"¿Todas son variables de entrada?: {set(DROPPED_COLS + TTL_COLS + LOG_COLS) <= set(INPUT_COLS)}")
print(f"Cruces de LOG_COLS o TTL_COLS con DROPPED_COLS: {sorted(set(LOG_COLS + TTL_COLS) & set(DROPPED_COLS))}")
print(f"Cruces de LOG_COLS con las categóricas: {sorted(set(LOG_COLS) & set(CATEGORICAL_COLS))}")
print(f"Cruces de TTL_COLS con LOG_COLS: {sorted(set(TTL_COLS) & set(LOG_COLS))}")

numeric = train_flows[[c for c in INPUT_COLS if c not in CATEGORICAL_COLS]]
skew = numeric.skew().round(2)
log_skew = np.log1p(numeric).skew().round(2)
eda_rule = skew.index[(numeric.nunique() > 20) & (skew > 1) & (log_skew.abs() < skew / 2)]
print(f"La regla del análisis exploratorio da {len(eda_rule)} variables")
print(f"¿Es LOG_COLS más stcpb y dtcpb?: {set(eda_rule) == set(LOG_COLS) | {'stcpb', 'dtcpb'}}")

## 4. Variables numéricas

`numeric_input_cols` da las variables numéricas que recibe el modelo: las variables de entrada menos las categóricas y las eliminadas, y sin las de TTL cuando `use_ttl=False`.

Comprobamos que son 32 con TTL y 29 sin TTL; que ninguna es eliminada, categórica, etiqueta ni `id`; que la lista sin TTL es la de con TTL menos `TTL_COLS`, en el mismo orden; que siguen el orden del archivo; que incluyen todas las de `LOG_COLS`, y que en la parte de entrenamiento todas son de tipo numérico.

In [ ]:
from threat_detector.dataset import ID_COL, TARGET_COLS
from threat_detector.ml.preprocessing import numeric_input_cols

with_ttl = numeric_input_cols(use_ttl=True)
without_ttl = numeric_input_cols(use_ttl=False)
not_allowed = set(DROPPED_COLS) | set(CATEGORICAL_COLS) | set(TARGET_COLS) | {ID_COL}

print(f"Con TTL: {len(with_ttl)} variables; sin TTL: {len(without_ttl)}")
print(f"Columnas que no deberían estar: {sorted(set(with_ttl) & not_allowed)}")
print(f"¿Sin TTL = con TTL menos TTL_COLS, en el mismo orden?: {without_ttl == [c for c in with_ttl if c not in TTL_COLS]}")
print(f"¿En el orden del archivo?: {with_ttl == [c for c in INPUT_COLS if c in with_ttl]}")
print(f"¿Incluyen todas las de LOG_COLS?: {set(LOG_COLS) <= set(without_ttl)}")
print(f"¿Todas numéricas en la parte de entrenamiento?: {all(pd.api.types.is_numeric_dtype(train_part[c]) for c in with_ttl)}")

## 5. Validación de flujos

Los datos del proyecto no tienen valores inválidos: la parte de validación y la muestra de demostración pasan la validación. Para probar que `validate_flows` sí detecta los errores que puede traer un CSV subido a la app, creamos **copias** de unos pocos flujos con errores puestos a propósito. Los datos originales no se modifican.

`validate_flows` comprueba, antes de preparar los flujos, que se pueden preparar. Revisa en este orden y se detiene en el primer problema: que haya al menos un flujo, que estén todas las columnas que usa el modelo (numéricas y categóricas), que las numéricas no tengan valores vacíos, que no son números o infinitos, y que no sean negativas. El mensaje dice cuántos flujos fallan, en qué columnas y en qué filas (como líneas del archivo, con el encabezado en la línea 1, máximo 5).

Las categóricas solo se exigen como columna: un valor vacío o desconocido no se rechaza, porque va a la columna de categorías infrecuentes.

Para los casos con valores inválidos tomamos 10 flujos de la mitad de la prueba **con su índice original** y ponemos el problema en las posiciones 3 y 7: el mensaje debe decir filas 5 y 9, no el índice. Así se ve que las filas se cuentan por posición, como en el archivo.

In [ ]:
from threat_detector.config import DEMO_SAMPLE_CSV, DEMO_SAMPLE_SIZE
from threat_detector.dataset import load_flows, validate_flows

numeric_cols = numeric_input_cols()


def show_validation(name, flows):
    """Valida una tabla y muestra si pasa o el mensaje de error."""
    try:
        validate_flows(flows, numeric_cols, CATEGORICAL_COLS)
        print(f"{name}: pasa")
    except ValueError as error:
        print(f"{name}: {error}")


middle = test_flows.iloc[41_166:41_176]


def with_values(column, value, positions=(3, 7)):
    """Copia de `middle` con `value` en las posiciones indicadas de `column`."""
    changed = middle.copy()
    changed[column] = changed[column].astype(object)
    changed.iloc[list(positions), changed.columns.get_loc(column)] = value
    return changed


print(f"Índice original de las posiciones 3 y 7: {middle.index[3]} y {middle.index[7]}\n")

print("Casos válidos")
show_validation("Parte de validación", validation_part)
show_validation("Muestra de demostración", load_flows(DEMO_SAMPLE_CSV, DEMO_SAMPLE_SIZE))
show_validation("Celda vacía en service", with_values("service", None))

print("\nCasos inválidos")
show_validation("Sin flujos", test_flows.iloc[:0])
show_validation("Sin sbytes, dur ni proto", validation_part.drop(columns=["sbytes", "dur", "proto"]))
show_validation("Celda vacía", with_values("sbytes", np.nan))
show_validation("Texto", with_values("sbytes", "abc"))
show_validation("Infinito", with_values("dur", np.inf))
show_validation("Negativo", with_values("dbytes", -5))
show_validation("8 negativos", with_values("sbytes", -1, positions=range(8)))

both = with_values("sbytes", -1, positions=[2])
both.iloc[1, both.columns.get_loc("dur")] = np.nan
show_validation("Vacío y negativo a la vez", both)

checked = with_values("sbytes", "abc")
snapshot = checked.copy()
show_validation("\nTabla con texto, otra vez", checked)
print(f"¿La tabla quedó igual después de validarla?: {checked.equals(snapshot)}")

## 6. Los tres preprocesadores

`build_preprocessor` arma el preprocesador de cada método de `PREPROCESSING_METHODS`:
- **M1:** `log1p` a `LOG_COLS`, las demás numéricas sin cambios y one-hot de las categóricas.
- **M2:** M1 más estandarización (`StandardScaler`) de todas las numéricas, después de `log1p`. Las columnas one-hot no se escalan.
- **M3:** M2 sin las variables de TTL.

Cada preprocesador se ajusta **solo con la parte de entrenamiento** y después transforma entrenamiento, validación y prueba, como se usará al entrenar: primero se validan los flujos con `validate_flows` y luego se transforman.

Comprobamos que en cada método las tres tablas tienen las mismas columnas en el mismo orden (53, 53 y 50), sin nombres repetidos ni prefijos (`sbytes`, no `log__sbytes`), sin vacíos ni infinitos, y solo con números.

In [ ]:
from threat_detector.ml.preprocessing import PREPROCESSING_METHODS, build_preprocessor

datasets = {"entrenamiento": train_part, "validación": validation_part, "prueba": test_flows}
preprocessors = {}
transformed = {}
for method, settings in PREPROCESSING_METHODS.items():
    for flows in datasets.values():
        validate_flows(flows, numeric_input_cols(settings["use_ttl"]), CATEGORICAL_COLS)
    preprocessors[method] = build_preprocessor(method).fit(train_part)
    transformed[method] = {name: preprocessors[method].transform(flows) for name, flows in datasets.items()}

summary = []
for method, tables in transformed.items():
    columns = [list(table.columns) for table in tables.values()]
    summary.append({
        "método": method,
        "columnas (entr., valid., prueba)": ", ".join(str(len(c)) for c in columns),
        "¿mismas columnas y orden?": columns[0] == columns[1] == columns[2],
        "¿nombres repetidos?": len(set(columns[0])) != len(columns[0]),
        "¿nombres con prefijo?": any("__" in name for name in columns[0]),
        "¿vacíos?": any(table.isna().any().any() for table in tables.values()),
        "¿infinitos?": not all(np.isfinite(table.to_numpy()).all() for table in tables.values()),
        "¿solo números?": all(pd.api.types.is_numeric_dtype(t) for table in tables.values() for t in table.dtypes),
    })

print(f"Primeras columnas de M1: {list(transformed['m1']['entrenamiento'].columns[:6])}")
pd.DataFrame(summary)

## 7. Categorías aprendidas

El análisis exploratorio conservó 18 categorías (las que tienen al menos `MIN_CATEGORY_COUNT` flujos en el archivo de entrenamiento completo); con la columna de categorías infrecuentes de cada variable quedan 21 columnas one-hot.

Comprobamos que el `OneHotEncoder` de cada método, ajustado con la parte de entrenamiento y `MIN_CATEGORY_FRACTION`, conservó **exactamente las mismas categorías** del análisis exploratorio y que sus columnas one-hot son 21.

In [ ]:
EDA_CATEGORIES = {
    "proto": ["arp", "ospf", "sctp", "tcp", "udp", "unas"],
    "service": ["-", "dns", "ftp", "ftp-data", "http", "pop3", "smtp", "ssh"],
    "state": ["CON", "FIN", "INT", "REQ"],
}

for method, preprocessor in preprocessors.items():
    encoder = preprocessor.named_transformers_["categorical"]
    kept = {
        col: [c for c in categories if c not in (infrequent if infrequent is not None else [])]
        for col, categories, infrequent in zip(CATEGORICAL_COLS, encoder.categories_, encoder.infrequent_categories_)
    }
    one_hot_cols = encoder.get_feature_names_out()
    print(f"{method}: ¿mismas categorías del análisis exploratorio?: {kept == EDA_CATEGORIES}; columnas one-hot: {len(one_hot_cols)}")

print(f"\nColumnas one-hot: {list(one_hot_cols)}")

## 8. Categorías nuevas, raras y vacías

Una categoría que no se conoce al ajustar (nueva), una que tiene muy pocos flujos en la parte de entrenamiento (rara) y una celda vacía no causan error: las tres van a la columna de categorías infrecuentes de su variable (`*_infrequent_sklearn`).

Tomamos copias de 4 flujos de la prueba y les ponemos un protocolo nuevo (`xyz`), un protocolo raro del entrenamiento, un servicio vacío y un estado nuevo (`ZZZ`). Comprobamos que pasan la validación, que se transforman sin error y que cada caso cae en su columna infrecuente. Además, comprobamos en las tablas de la sección 6 que cada flujo tiene exactamente un 1 por variable categórica.

In [ ]:
m1_encoder = preprocessors["m1"].named_transformers_["categorical"]
rare_proto = m1_encoder.infrequent_categories_[0][0]

unusual = middle.iloc[:4].copy()
for col in CATEGORICAL_COLS:
    unusual[col] = unusual[col].astype(object)
cases = [
    ("Protocolo nuevo (xyz)", "proto", "xyz"),
    (f"Protocolo raro ({rare_proto})", "proto", rare_proto),
    ("Servicio vacío", "service", None),
    ("Estado nuevo (ZZZ)", "state", "ZZZ"),
]
for position, (_, col, value) in enumerate(cases):
    unusual.iloc[position, unusual.columns.get_loc(col)] = value

show_validation("Flujos con categorías nuevas, raras y vacías", unusual)
unusual_result = preprocessors["m1"].transform(unusual)
for position, (name, col, _) in enumerate(cases):
    print(f"{name}: {col}_infrequent_sklearn = {unusual_result.iloc[position][f'{col}_infrequent_sklearn']:.0f}")

one_per_variable = all(
    (table.loc[:, table.columns.str.startswith(f"{col}_")].sum(axis=1) == 1).all()
    for tables in transformed.values()
    for table in tables.values()
    for col in CATEGORICAL_COLS
)
print(f"\n¿Exactamente un 1 por variable categórica en todas las tablas?: {one_per_variable}")

## 9. `log1p`

En M1 (sin escalar) se ve directamente el efecto de `log1p`: cada variable de `LOG_COLS` debe ser igual a `np.log1p` de su valor original, y las numéricas sin `log1p` deben quedar exactamente como venían.

In [ ]:
m1_train = transformed["m1"]["entrenamiento"]
m1_numeric = numeric_input_cols(use_ttl=True)
plain_cols = [c for c in m1_numeric if c not in LOG_COLS]

log_ok = all(np.array_equal(m1_train[c].to_numpy(), np.log1p(train_part[c]).to_numpy()) for c in LOG_COLS)
plain_ok = all(np.array_equal(m1_train[c].to_numpy(), train_part[c].to_numpy()) for c in plain_cols)
print(f"¿Las {len(LOG_COLS)} variables de LOG_COLS son log1p del original?: {log_ok}")
print(f"¿Las {len(plain_cols)} numéricas sin log1p quedaron iguales?: {plain_ok}")
print(f"Ejemplo, primer flujo: sbytes = {train_part['sbytes'].iloc[0]} -> {m1_train['sbytes'].iloc[0]:.4f}; sttl = {train_part['sttl'].iloc[0]} -> {m1_train['sttl'].iloc[0]}")

## 10. Escalado (M2 y M3)

En M2 y M3 todas las numéricas se estandarizan después de `log1p`: en la parte de entrenamiento deben quedar con media 0 y desviación 1. Las columnas one-hot no se escalan y deben seguir siendo 0 y 1.

También comprobamos de dónde salió lo aprendido: la media que guardó el escalador de las variables con `log1p` debe ser la de la parte de entrenamiento (ya con `log1p`), y no la de la validación. Por último, comprobamos que solo M2 y M3 tienen escalador.

In [ ]:
from sklearn.preprocessing import StandardScaler

for method in ("m2", "m3"):
    table = transformed[method]["entrenamiento"]
    numeric = numeric_input_cols(PREPROCESSING_METHODS[method]["use_ttl"])
    one_hot_values = set(np.unique(table.drop(columns=numeric).to_numpy()))
    scaled_log_cols = [c for c in numeric if c in LOG_COLS]
    scaler = preprocessors[method].named_transformers_["log"][-1]
    train_mean = np.log1p(train_part[scaled_log_cols]).mean().to_numpy()
    validation_mean = np.log1p(validation_part[scaled_log_cols]).mean().to_numpy()

    print(f"{method}: media más lejana de 0 = {table[numeric].mean().abs().max():.1e}; "
          f"desviación más lejana de 1 = {(table[numeric].std(ddof=0) - 1).abs().max():.1e}; "
          f"valores one-hot = {sorted(one_hot_values)}")
    print(f"    ¿Media del escalador = la de entrenamiento?: {np.allclose(scaler.mean_, train_mean)}; "
          f"¿= la de validación?: {np.allclose(scaler.mean_, validation_mean)}")

print()
for method, preprocessor in preprocessors.items():
    pieces = [preprocessor.named_transformers_["log"][-1], preprocessor.named_transformers_["plain"]]
    print(f"{method}: ¿tiene escalador?: {any(isinstance(piece, StandardScaler) for piece in pieces)}")

## 11. M3 sin TTL

M3 es M2 sin las variables de TTL. Comprobamos que no tiene ninguna de `TTL_COLS` y que el resto de sus columnas son las de M2, en el mismo orden.

In [ ]:
m2_cols = list(transformed["m2"]["entrenamiento"].columns)
m3_cols = list(transformed["m3"]["entrenamiento"].columns)

print(f"Variables de TTL en M3: {sorted(set(TTL_COLS) & set(m3_cols))}")
print(f"¿M3 = M2 menos TTL_COLS, en el mismo orden?: {m3_cols == [c for c in m2_cols if c not in TTL_COLS]}")

## 12. La tabla original no cambia

El preprocesador devuelve una tabla nueva y nunca modifica la que recibe. Guardamos una copia de cada tabla, la transformamos otra vez con los tres métodos y comprobamos que sigue igual a su copia.

In [ ]:
snapshots = {name: flows.copy() for name, flows in datasets.items()}
for preprocessor in preprocessors.values():
    for flows in datasets.values():
        preprocessor.transform(flows)

for name, flows in datasets.items():
    print(f"{name}: ¿igual a su copia?: {flows.equals(snapshots[name])}")

## 13. Guardar y cargar con `joblib`

Cada preprocesador ajustado se guarda dentro de `model.joblib`, así que debe dar exactamente lo mismo después de guardarlo y cargarlo. Lo guardamos en una carpeta temporal (que se borra al terminar), lo cargamos y comprobamos que transforma la validación y la prueba igual que antes.

Además, el contrato pide que para cargarlo basten scikit-learn y joblib, sin importar `threat_detector`. Lo comprobamos cargando M2 en un proceso de Python aparte y revisando si se importó `threat_detector`.

In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

import joblib

check_imports = (
    "import sys, joblib; joblib.load(sys.argv[1]); "
    "print('threat_detector' in sys.modules)"
)

with tempfile.TemporaryDirectory() as temp_dir:
    for method, preprocessor in preprocessors.items():
        path = Path(temp_dir) / f"{method}.joblib"
        joblib.dump(preprocessor, path)
        loaded = joblib.load(path)
        same = all(
            loaded.transform(flows).equals(transformed[method][name])
            for name, flows in (("validación", validation_part), ("prueba", test_flows))
        )
        print(f"{method}: {path.stat().st_size / 1024:.1f} KB; ¿mismo resultado después de cargarlo?: {same}")

    imported = subprocess.run(
        [sys.executable, "-c", check_imports, str(Path(temp_dir) / "m2.joblib")],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    print(f"\n¿Cargar M2 en otro proceso importó threat_detector?: {imported}")

print(f"¿Se borró la carpeta temporal?: {not Path(temp_dir).exists()}")